In [1]:
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag,ResultSet
import random
import pandas as pd
import time
import re
from datatypes import JobPosting

Starts at 25 and then you get more jobs

In [3]:
title=""
location = ""
url = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords=Machine%20Learning%20Engineer&location=Washington%2C%20District%20of%20Columbia%2C%20United%20States&locationId=&geoId=104383890&f_TPR=&distance=25&f_SB2=5&position=1&pageNum=0&start=25"

In [4]:
request=requests.get(url)
print(request)
list_data=request.text
list_soup = BeautifulSoup(list_data, 'html.parser')
page_jobs = list_soup.find_all('li')

job_id_list = []
for job in page_jobs:
    base_card_div = job.find('div', class_='base-card')
    job_id = base_card_div['data-entity-urn'].split(":")[-1]
    job_id_list.append(job_id)

<Response [200]>


In [5]:
def parse_job_description(job_soup:BeautifulSoup) -> str:
    formatted_text_pieces = []
    job_description_tag=job_soup.find("div", {"class": "show-more-less-html__markup"})
    # Extract and format <strong> elements and following siblings until the next <strong> element or list
    for child in job_description_tag.children:
        if child.name == "strong":
                # Format headings
                formatted_text_pieces.append(f"{child.get_text().strip()}\n")
        elif child.name == "p":
            # Format paragraphs, handling <br> tags as new lines
            text = child.get_text(separator="\n", strip=True)
            formatted_text_pieces.append(text)
        elif child.name in ["ul", "ol"]:
            # Format lists
            list_items = [li.get_text(separator="\n", strip=True) for li in child.find_all("li")]
            formatted_list = "\n".join([f"- {item}" for item in list_items])
            formatted_text_pieces.append(formatted_list)
        elif child.name == "br":
            # Optionally, handle breaks if needed
            continue
        else:
            # Handle other types of elements as needed or ignore
            text = child.get_text(separator="\n", strip=True)
            if text:
                formatted_text_pieces.append(text)

    # Join the formatted text pieces into a single string
    formatted_text = "\n\n".join(formatted_text_pieces)

    return formatted_text

def parse_job_criteria(job_soup: BeautifulSoup) -> dict:
    criteria_dict = {}
    job_criteria = job_soup.find_all("li", class_="description__job-criteria-item")
    # Extracting and printing each component
    for item in job_criteria:
        # Extract the criteria name and detail
        criteria_name = item.find("h3", class_="description__job-criteria-subheader").text.strip()
        criteria_detail = item.find("span", class_="description__job-criteria-text description__job-criteria-text--criteria").text.strip()

        # Add to the dictionary
        criteria_dict[criteria_name] = criteria_detail

    return criteria_dict

def extract_job_poster(job_soup: BeautifulSoup) -> dict:
    name = ""
    profile_url = ""
    message_recruiter_div = job_soup.find("div", class_="message-the-recruiter")
    if message_recruiter_div:
        # If the div exists, find the <a> tag for the URL and name
        profile_tag = message_recruiter_div.find("a", class_="base-card__full-link absolute top-0 right-0 bottom-0 left-0 p-0 z-[2]")
        # Find the <h4> tag within the div for the detailed title
        detail_title_tag = message_recruiter_div.find("h4", class_="base-main-card__subtitle body-text text-color-text overflow-hidden")
        
        if profile_tag and detail_title_tag:
            # Extract the name, which is also visually hidden but available for screen readers
            name = profile_tag.find("span", class_="sr-only").text.strip() if profile_tag.find("span", class_="sr-only") else None
            # Extract the profile URL
            profile_url = profile_tag.get('href', None).split("?trk")[0] if profile_tag.get('href', None) else None

            # Extract the title
            title = detail_title_tag.text.strip()
            # Compile the extracted information
            info = {
                "name": name,
                "title": title,
                "profile_url": profile_url
            }
            return info
    else:
        return None

def extract_salaries(job_description):
    # This pattern focuses on:
    # - Optional dollar sign
    # - Numbers potentially starting with $, possibly with a comma or period for thousands, and may end with "K" for thousands or specify "USD"
    # - Ignores solitary small numbers that are unlikely to represent salaries

    pattern = r'\$\d{1,3}(?:[.,]\d{3})*(?:-\$\d{1,3}(?:[.,]\d{3})*)?\s?(?:k|K|USD|usd)?|\d{1,3}(?:[.,]\d{3})+(?:-\d{1,3}(?:[.,]\d{3})*)?\s?(?:k|K|USD|usd)'

    
    # Find all matches in the job description
    matches = re.findall(pattern, job_description)
    
    # Process matches to standardize format (optional, depending on your needs)
    standardized_matches = [match.replace(',', '').replace('.', '').upper() for match in matches]
    
    return standardized_matches

In [7]:
for job_id in job_id_list:
    job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"
    print(f"Scraping job at {job_url}")
    while True:
        try:
            wait_time = random.randint(1, 5)
            print(f"Waiting {wait_time} seconds")
            time.sleep(wait_time)
            job_request = requests.get(job_url)
            if job_request.status_code == 429:
                print("Too many requests, waiting 10 seconds before retrying...")
                time.sleep(10)
                continue
            job_data = job_request.text
            job_soup = BeautifulSoup(job_data, 'html.parser')
            # create a string from the HTML
            job_title = job_soup.find("h2", {"class": "top-card-layout__title font-sans text-lg papabear:text-xl font-bold leading-open text-color-text mb-0 topcard__title"}).text.strip()
            company_name = job_soup.find("a", {"class": "topcard__org-name-link topcard__flavor--black-link"}).text.strip()
            company_url = job_soup.find("a", {"class": "topcard__org-name-link topcard__flavor--black-link"})['href'].split("?trk")[0]
            job_location = job_soup.find("span", {"class": "topcard__flavor topcard__flavor--bullet"}).text.strip()
            job_description_text=parse_job_description(job_soup)
            # print(job_description_text)
            job_criteria_dict = parse_job_criteria(job_soup)
            job_poster_info = extract_job_poster(job_soup)
            salary = extract_salaries(job_description_text)

            job_posting = JobPosting(
                job_id=job_id,
                title=job_title,
                seniority_level=job_criteria_dict.get("Seniority level", None),
                employment_type=job_criteria_dict.get("Employment type", None),
                job_description=job_description_text,
                company=company_name,
                company_url=company_url,
                job_functions=job_criteria_dict.get("Job function", None),
                industries=job_criteria_dict.get("Industries", None),
            )
            if job_poster_info:
                job_posting.job_poster_profile_url = job_poster_info.get("profile_url", None)
                job_posting.job_poster_name = job_poster_info.get("name", None)
            print(job_posting.to_json())
            

            break
        except Exception as e:
            print(f"An error occurred: {e}")
            break

    

Scraping job at https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/3739558642
Waiting 4 seconds
{
    "job_id": "3739558642",
    "title": "Junior Software Engineer",
    "seniority_level": "Entry level",
    "employment_type": "Full-time",
    "job_description": "Required Security Clearance\n\n\nTop Secret/SCI with Full Scope Polygraph\n\n\nCity\n\n\nFort Meade\n\n\nState/Territory\n\n\nMaryland\n\n\nTravel\n\n\nNone\n\n\nPotential For Teleworking\n\n\nYes\n\n\nSchedule\n\n\nFull Time\n\n\nDoD 8570 IAT Requirement\n\n\nNone\n\n\nDoD 8570 IAM Requirement\n\n\nNone\n\n\nDoD 8570 IASAE Requirement\n\n\nNone\n\n\nDoD CSSP Requirement\n\n\nNone\n\n\nLast Updated\n\n\n3/25/24 9:00 PM\n\n\nRequisition ID\n\n\n58176\n\n\nUS Citizenship Required?:\n\nYes\n\n\nAnnual Salary Range\n\n\n$110,000 to $140,000 with Full Benefits to include Health/Dental/Vision and PTO.\n\nDescription\n\n\nBase-2 Solutions is looking for a top-notch software engineer to join our team. We are driven to solve chal